# Extension 3: NIH Training Grants, Fellowships, and the Labor Advantage

**Builds on:** Zhang et al. (2022), "Labor advantages drive the greater productivity 
of faculty at elite universities", *Science Advances* 8, eabq7056.

**Key questions (from David V. Smith, Temple University):**
1. Are places like Temple seeing a less-than-expected productivity bump from funded 
   labor? (i.e., funded students without research expectations)
2. Does the productivity gap shrink once you decompose funding into intramural 
   (university fellowships/RAships) vs. extramural (R01 RA lines) vs. training grants 
   (F31, F32, T32)?
3. Can we build a "quality-weighted doctorate" metric that accounts for stipend, 
   lead-author publications, and funded research time?

**Requirements:** `pip install requests pandas numpy matplotlib seaborn statsmodels`

**Network access needed:** This notebook queries the NIH RePORTER API 
(`api.reporter.nih.gov`). It also works with bulk ExPORTER CSV downloads as a 
fallback. No API key is required.

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COLORS = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f']

# Create output directories
Path('data').mkdir(exist_ok=True)
Path('figures').mkdir(exist_ok=True)

# Load Zhang et al. data
DATA_DIR = './code-and-data/'
try:
    area_strict = pd.read_csv(f'{DATA_DIR}area-strict.csv')
    area_nonstrict = pd.read_csv(f'{DATA_DIR}area-non-strict.csv')
    print(f"Zhang et al. data loaded: {len(area_strict)} strict, {len(area_nonstrict)} non-strict departments")
except FileNotFoundError:
    print("WARNING: Zhang et al. data not found. Place code-and-data/ in the same directory.")
    area_strict = None

Zhang et al. data loaded: 739 strict, 1800 non-strict departments


---
## Part 1: Pull NIH Training Grant Data via RePORTER API

The NIH RePORTER API is free, requires no API key, and returns structured JSON.
We'll query all T32, F31, and F32 awards at every institution, then link to the 
Zhang et al. prestige data.

In [2]:
# ============================================================
# NIH RePORTER API FUNCTIONS
# ============================================================

REPORTER_API = "https://api.reporter.nih.gov/v2/projects/search"

def query_reporter(criteria, limit=500, offset=0, max_retries=3):
    """Query NIH RePORTER API. Returns list of project dicts."""
    payload = {
        "criteria": criteria,
        "limit": limit,
        "offset": offset,
        "sort_field": "project_start_date",
        "sort_order": "desc"
    }
    
    for attempt in range(max_retries):
        try:
            resp = requests.post(REPORTER_API, json=payload, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            return data.get('results', []), data.get('meta', {}).get('total', 0)
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # exponential backoff
            else:
                print(f"  API error after {max_retries} attempts: {e}")
                return [], 0

def query_all_pages(criteria, limit=500):
    """Page through all results for a query."""
    all_results = []
    offset = 0
    results, total = query_reporter(criteria, limit=limit, offset=offset)
    all_results.extend(results)
    
    while len(all_results) < total:
        offset += limit
        results, _ = query_reporter(criteria, limit=limit, offset=offset)
        if not results:
            break
        all_results.extend(results)
        time.sleep(0.3)  # be nice to the API
    
    return all_results

def projects_to_dataframe(projects):
    """Convert RePORTER project list to a clean DataFrame."""
    rows = []
    for p in projects:
        org = p.get('organization', {}) or {}
        pi_list = p.get('principal_investigators', []) or []
        pi_name = pi_list[0].get('full_name', '') if pi_list else ''
        
        rows.append({
            'project_num': p.get('project_num', ''),
            'activity_code': p.get('activity_code', ''),
            'fiscal_year': p.get('fiscal_year', None),
            'project_title': p.get('project_title', ''),
            'org_name': org.get('org_name', ''),
            'org_city': org.get('org_city', ''),
            'org_state': org.get('org_state', ''),
            'dept_type': org.get('dept_type', ''),
            'pi_name': pi_name,
            'award_amount': p.get('award_amount', 0),
            'project_start_date': p.get('project_start_date', ''),
            'project_end_date': p.get('project_end_date', ''),
            'ic_name': (p.get('agency_ic_fundings', [{}]) or [{}])[0].get('name', '') if p.get('agency_ic_fundings') else '',
        })
    return pd.DataFrame(rows)

In [3]:
# ============================================================
# QUERY 1: All T32, F31, F32 awards in Psychology-related departments
# ============================================================

print("Querying NIH RePORTER for training awards...")
print("(This may take a few minutes for large queries)")

# We'll query fiscal years 2008-2024 to span Zhang et al.'s period and extend it
training_data = {}

for activity_code in ['T32', 'F31', 'F32']:
    print(f"\n--- Querying {activity_code} awards ---")
    criteria = {
        "activity_codes": [activity_code],
        "fiscal_years": list(range(2008, 2025)),
        # Broad query - we'll filter by department later
    }
    
    results = query_all_pages(criteria)
    print(f"  Retrieved {len(results)} {activity_code} awards")
    
    if results:
        df = projects_to_dataframe(results)
        training_data[activity_code] = df
        df.to_csv(f'data/{activity_code}_awards_2008_2024.csv', index=False)
        print(f"  Saved to data/{activity_code}_awards_2008_2024.csv")
    else:
        print(f"  No results (API may be unreachable - see fallback below)")

# If API is unreachable, provide instructions for bulk download
if not any(training_data.values()):
    print("""
╔══════════════════════════════════════════════════════════════╗
║  API UNREACHABLE - USE BULK DOWNLOAD INSTEAD                ║
║                                                              ║
║  1. Go to: https://reporter.nih.gov/exporter                ║
║  2. Download "Projects" CSV files for years 2008-2024       ║
║  3. Place them in the data/ directory                        ║
║  4. Run the cell below to load from CSV                      ║
║                                                              ║
║  Alternatively, use the RePORTER web interface:              ║
║  https://reporter.nih.gov/                                   ║
║  Search for Activity Code = T32, F31, or F32                 ║
║  Export results as CSV                                        ║
╚══════════════════════════════════════════════════════════════╝
    """)

Querying NIH RePORTER for training awards...
(This may take a few minutes for large queries)

--- Querying T32 awards ---
  API error after 3 attempts: 400 Client Error: Bad Request for url: https://api.reporter.nih.gov/v2/projects/search
  Retrieved 15000 T32 awards
  Saved to data/T32_awards_2008_2024.csv

--- Querying F31 awards ---
  API error after 3 attempts: 400 Client Error: Bad Request for url: https://api.reporter.nih.gov/v2/projects/search
  Retrieved 15000 F31 awards
  Saved to data/F31_awards_2008_2024.csv

--- Querying F32 awards ---
  API error after 3 attempts: 400 Client Error: Bad Request for url: https://api.reporter.nih.gov/v2/projects/search
  Retrieved 15000 F32 awards
  Saved to data/F32_awards_2008_2024.csv


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))
  Retrieved 0 F31 awards
  No results (API may be unreachable - see fallback below)

--- Querying F32 awards ---


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))
  Retrieved 0 F32 awards
  No results (API may be unreachable - see fallback below)

╔══════════════════════════════════════════════════════════════╗
║  API UNREACHABLE - USE BULK DOWNLOAD INSTEAD                ║
║                                                              ║
║  1. Go to: https://reporter.nih.gov/exporter                ║
║  2. Download "Projects" CSV files for years 2008-2024       ║
║  3. Place them in the data/ directory                        ║
║  4. Run the cell below to load from CSV                      ║
║                                                              ║
║  Alternatively, use the RePORTER web interface:              ║
║  https://reporter.nih.gov/                                   ║
║  Search for Activity

In [ ]:
# ============================================================
# FALLBACK: Load from ExPORTER bulk CSV downloads
# ============================================================

def load_exporter_csvs(data_dir='data'):
    """Load from NIH ExPORTER bulk CSV files if API was unreachable."""
    import glob
    
    csv_files = glob.glob(f'{data_dir}/RePORTER_PRJ_C_FY*.csv')
    if not csv_files:
        print("No ExPORTER CSV files found in data/")
        print("Download from: https://reporter.nih.gov/exporter")
        return None
    
    dfs = []
    for f in sorted(csv_files):
        print(f"Loading {f}...")
        df = pd.read_csv(f, low_memory=False, encoding='latin-1')
        # Filter to training grants only
        df = df[df['ACTIVITY'].isin(['T32', 'F31', 'F32'])]
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(combined)} training awards from {len(csv_files)} files")
    return combined

# Uncomment and run this if the API was unreachable:
# exporter_data = load_exporter_csvs('data')

In [ ]:
# ============================================================
# QUERY 2: Focus on Temple University specifically
# ============================================================

print("\n=== TEMPLE UNIVERSITY TRAINING AWARDS ===")
print()

for activity_code in ['T32', 'F31', 'F32']:
    criteria = {
        "org_names": ["TEMPLE UNIVERSITY"],
        "activity_codes": [activity_code],
        "fiscal_years": list(range(2000, 2025)),
    }
    
    results, total = query_reporter(criteria, limit=500)
    
    if results:
        df = projects_to_dataframe(results)
        
        print(f"\n{activity_code} at Temple University ({total} total):")
        print(f"  Fiscal years: {df['fiscal_year'].min()}-{df['fiscal_year'].max()}")
        print(f"  Departments: {df['dept_type'].value_counts().to_dict()}")
        print(f"  Unique projects: {df['project_num'].nunique()}")
        
        # Show psychology-related
        psych = df[df['dept_type'].str.contains('PSYCHO|NEURO|BRAIN', case=False, na=False)]
        if len(psych) > 0:
            print(f"  Psychology/Neuro related: {len(psych)} awards")
            for _, row in psych.drop_duplicates('project_num').iterrows():
                print(f"    {row['project_num']}: {row['project_title'][:70]}...")
    else:
        print(f"\n{activity_code} at Temple: API unreachable or no results")
        print(f"  Manual check: https://reporter.nih.gov/ → search Temple University + {activity_code}")

  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))

T32 at Temple: API unreachable or no results
  Manual check: https://reporter.nih.gov/ → search Temple University + T32


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))

F31 at Temple: API unreachable or no results
  Manual check: https://reporter.nih.gov/ → search Temple University + F31


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))

F32 at Temple: API unreachable or no results
  Manual check: https://reporter.nih.gov/ → search Temple University + F32


In [ ]:
# ============================================================
# QUERY 3: Comparison institutions in similar prestige band
# ============================================================
# These are ~R1 universities in the same approximate prestige range as Temple
# for Psychology specifically (mid-tier R1s)

COMPARISON_INSTITUTIONS = [
    "TEMPLE UNIVERSITY",
    "UNIVERSITY OF PITTSBURGH AT PITTSBURGH",
    "PENNSYLVANIA STATE UNIVERSITY",
    "DREXEL UNIVERSITY",
    "RUTGERS, THE STATE UNIVERSITY OF NEW JERSEY",
    "UNIVERSITY OF DELAWARE",
    "GEORGE MASON UNIVERSITY",
    "VIRGINIA COMMONWEALTH UNIVERSITY",
    "UNIVERSITY OF CONNECTICUT",
    "UNIVERSITY OF GEORGIA",
    # Add some clearly higher-prestige comparisons
    "UNIVERSITY OF PENNSYLVANIA",
    "YALE UNIVERSITY",
    "COLUMBIA UNIVERSITY HEALTH SCIENCES",
    "UNIVERSITY OF MICHIGAN AT ANN ARBOR",
]

comparison_results = []

for inst in COMPARISON_INSTITUTIONS:
    for activity_code in ['T32', 'F31', 'F32']:
        criteria = {
            "org_names": [inst],
            "activity_codes": [activity_code],
            "fiscal_years": list(range(2015, 2025)),
            # Filter to psych/neuro departments
            "dept_types": ["PSYCHOLOGY", "NEUROSCIENCES", "PSYCHIATRY",
                          "BEHAVIORAL SCIENCES", "MENTAL HEALTH"]
        }
        
        results, total = query_reporter(criteria, limit=500)
        
        comparison_results.append({
            'institution': inst,
            'activity_code': activity_code,
            'n_awards': total,
            'n_unique_projects': len(set(r.get('project_num','') for r in results)) if results else 0,
            'total_funding': sum(r.get('award_amount', 0) or 0 for r in results) if results else 0,
        })
        time.sleep(0.2)

comparison_df = pd.DataFrame(comparison_results)
print("\nTraining Awards in Psychology/Neuro Departments (2015-2024):")
pivot = comparison_df.pivot_table(
    index='institution', columns='activity_code', 
    values='n_awards', fill_value=0
)
print(pivot.to_string())

  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))


...

  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))


  API error after 3 attempts: HTTPSConnectionPool(host='api.reporter.nih.gov', port=443): Max retries exceeded with url: /v2/projects/search (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))

Training Awards in Psychology/Neuro Departments (2015-2024):


activity_code                                F31  F32  T32
institution                                               
COLUMBIA UNIVERSITY HEALTH SCIENCES          0.0  0.0  0.0
DREXEL UNIVERSITY                            0.0  0.0  0.0
GEORGE MASON UNIVERSITY                      0.0  0.0  0.0
PENNSYLVANIA STATE UNIVERSITY                0.0  0.0  0.0
RUTGERS, THE STATE UNIVERSITY OF NEW JERSEY  0.0  0.0  0.0
TEMPLE UNIVERSITY                            0.0  0.0  0.0
UNIVERSITY OF CONNECTICUT                    0.0  0.0  0.0
UNIVERSITY OF DELAWARE                       0.0  0.0  0.0
UNIVERSITY OF GEORGIA                        0.0  0.0  0.0
UNIVERSITY OF MICHIGAN AT ANN ARBOR          0.0  0.0  0.0
UNIVERSITY OF PENNSYLVANIA                   0.0  0.0  0.0
UNIVERSITY OF PITTSBURGH AT PITTSBURGH       0.0  0.0  0.0
VIRGINIA COMMONWEALTH UNIVERSITY             0.0  0.0  0.0
YALE UNIVERSITY                              0.0  0.0  0.0


---
## Part 2: Linking Training Grants to the Labor Advantage

### Question 1: Is Temple seeing less-than-expected productivity from funded labor?

The hypothesis: At institutions where funded graduate students have **no additional 
research expectations** (i.e., they're on departmental fellowships with teaching 
obligations, not on PI grants), the "funded labor" in the NSF GSS data doesn't 
translate to actual research labor. The Zhang et al. model would overpredict 
productivity for these departments.

In [ ]:
# ============================================================
# ANALYSIS: Funded labor → productivity residuals by training grant density
# ============================================================

# This analysis requires linking the NIH data to Zhang et al.'s department data.
# Since Zhang's data is anonymized, we create the linkage through the NSF GSS
# public use files (which have institution identifiers).

# Step 1: Build institution-level training grant counts
def build_institution_training_profile(comparison_df):
    """Aggregate training grant counts per institution."""
    profile = comparison_df.groupby('institution').agg({
        'n_awards': 'sum',
        'total_funding': 'sum'
    }).reset_index()
    
    # Pivot by activity code
    by_type = comparison_df.pivot_table(
        index='institution', columns='activity_code',
        values='n_awards', fill_value=0, aggfunc='sum'
    ).reset_index()
    
    profile = profile.merge(by_type, on='institution')
    profile['training_intensity'] = profile['T32'] / (profile['F31'] + profile['F32'] + 1)
    profile['fellowship_rate'] = (profile['F31'] + profile['F32']) / (profile['n_awards'] + 1)
    
    return profile

if len(comparison_df) > 0 and comparison_df['n_awards'].sum() > 0:
    inst_profiles = build_institution_training_profile(comparison_df)
    print("Institution Training Profiles (Psychology/Neuro, 2015-2024):")
    print(inst_profiles.to_string(index=False))
else:
    print("No comparison data available (API unreachable)")
    print("Run this notebook locally with internet access to populate.")

### Question 2: Decomposing the funding sources

Zhang et al. treat all "funded researchers" as equivalent. But there are (at least) 
three distinct types of funded labor, each with different implications:

| Funding Type | Source | Research Expectation | PI Control |
|---|---|---|---|
| University/dept fellowship | Intramural | Often low (teaching focus) | Low |
| RA line on R01/R21 | PI's grant | High (PI-directed research) | High |
| T32 trainee slot | NIH training grant | High (structured program) | Moderate |
| F31/F32 fellow | Individual NIH award | Very high (self-directed) | Low-Moderate |

**The key insight:** The NSF GSS counts students by *mechanism* of support (RA, 
fellowship, traineeship, TA, self-funded) — but it doesn't distinguish *intramural* 
fellowships (which may have no research expectations) from *extramural* fellowships 
(T32/F31, which have strong research expectations).

This is exactly David's point: at Temple, funded students may be on departmental 
fellowships with no research obligations, which means the NSF GSS "funded" count 
overstates the actual research labor available.

In [4]:
# ============================================================
# ANALYTICAL FRAMEWORK: Decomposing the labor advantage
# ============================================================

print("""
PROPOSED REGRESSION MODEL (Extension of Zhang et al. Table S1):
═══════════════════════════════════════════════════════════════

Original Zhang et al. model:
  Productivity ~ β₁·FundedLabor + β₂·Prestige + β₃·UnfundedLabor + controls

Proposed decomposition (requires NSF GSS + NIH RePORTER linkage):
  Productivity ~ β₁·RA_on_grants + β₂·T32_trainees + β₃·F_fellows 
                + β₄·Intramural_fellows + β₅·Prestige + controls

PREDICTIONS:
  If David's hypothesis is correct:
  • β₁ (RA on grants) >> β₄ (intramural fellows) 
    → RA-funded students are more productive because they MUST do research
  • β₂ (T32 trainees) > β₄ (intramural fellows)
    → T32 students have structured research training requirements
  • β₃ (F31/F32) ≈ β₂ (T32) or higher
    → Individually competitive fellows are at least as productive
  • β₄ (intramural fellows) may even be ≈ 0 or negative
    → If these students primarily teach and don't contribute to PI publications

DATA NEEDED:
  1. NSF GSS public use files (has mechanism of support breakdown)
     → Download from: https://ncses.nsf.gov/explore-data/microdata/graduate-students-postdoctorates-s-e
  2. NIH RePORTER data (T32/F31/F32 counts per institution-department)
     → From Part 1 of this notebook
  3. Zhang et al. department data (productivity + prestige)
     → Already loaded
  4. IPEDS (institution crosswalk IDs)
     → Download from: https://nces.ed.gov/ipeds/
""")


PROPOSED REGRESSION MODEL (Extension of Zhang et al. Table S1):
═══════════════════════════════════════════════════════════════

Original Zhang et al. model:
  Productivity ~ β₁·FundedLabor + β₂·Prestige + β₃·UnfundedLabor + controls

Proposed decomposition (requires NSF GSS + NIH RePORTER linkage):
  Productivity ~ β₁·RA_on_grants + β₂·T32_trainees + β₃·F_fellows 
                + β₄·Intramural_fellows + β₅·Prestige + controls

PREDICTIONS:
  If David's hypothesis is correct:
  • β₁ (RA on grants) >> β₄ (intramural fellows) 
    → RA-funded students are more productive because they MUST do research
  • β₂ (T32 trainees) > β₄ (intramural fellows)
    → T32 students have structured research training requirements
  • β₃ (F31/F32) ≈ β₂ (T32) or higher
    → Individually competitive fellows are at least as productive
  • β₄ (intramural fellows) may even be ≈ 0 or negative
    → If these students primarily teach and don't contribute to PI publications

DATA NEEDED:
  1. NSF GSS public use

### Question 3: Quality-Weighted Doctorate Metric

The Carnegie R1 classification uses raw doctorate counts. This is, as David notes,
a crude metric. Here's a proposed "research-weighted doctorate" metric.

In [5]:
print("""
PROPOSED: Research-Weighted Doctorate (RWD) Metric
═══════════════════════════════════════════════════

For each doctoral graduate i at institution j:

  RWD_i = α₁ · (first-author publications during training)
        + α₂ · (funded research months / total months in program)
        + α₃ · (stipend_i / median_stipend_field)
        + α₄ · (1 if postdoc placement at R1, 0.5 if industry research, 0 otherwise)

Then aggregate: RWD_j = Σ RWD_i for all graduates at institution j

DATA SOURCES:
─────────────
1. First-author publications:
   → OpenAlex API (free, open): query by author ORCID or name + affiliation
   → Filter to papers where author position = first
   → This is the MOST IMPORTANT component and is freely available

2. Funded research months:
   → NSF GSS: mechanism of support (RA/fellowship/TA) by year
   → NIH RePORTER: exact funding periods for T32/F31/F32
   → ProQuest dissertations: time-to-degree (proxy for research intensity)
   
3. Stipend:
   → NSF GSS doesn't report stipends directly
   → NIH NRSA stipend scale is public and uniform
   → AAUP salary surveys + institutional data for non-NIH stipends
   → PROXY: use institution's total R&D expenditure per grad student
   
4. Postdoc placement:
   → Track graduates through OpenAlex affiliation changes
   → ProQuest Dissertations (advisor + institution)
   → Academic Analytics (if available under DUA)

FEASIBILITY RANKING:
  Component 1 (publications): ★★★★★ Easy, free via OpenAlex
  Component 2 (funded months): ★★★☆☆ Medium, requires GSS + RePORTER linkage  
  Component 3 (stipend):       ★★☆☆☆ Hard, mostly unavailable at individual level
  Component 4 (placement):     ★★★☆☆ Medium, trackable via OpenAlex

RECOMMENDATION: Start with components 1 and 4 only (both from OpenAlex).
  Simplified RWD_i = (first-author pubs) × (1 + 0.5 × postdoc_at_R1)
  This is computable RIGHT NOW with zero restricted data.
""")


PROPOSED: Research-Weighted Doctorate (RWD) Metric
═══════════════════════════════════════════════════

For each doctoral graduate i at institution j:

  RWD_i = α₁ · (first-author publications during training)
        + α₂ · (funded research months / total months in program)
        + α₃ · (stipend_i / median_stipend_field)
        + α₄ · (1 if postdoc placement at R1, 0.5 if industry research, 0 otherwise)

Then aggregate: RWD_j = Σ RWD_i for all graduates at institution j

DATA SOURCES:
─────────────
1. First-author publications:
   → OpenAlex API (free, open): query by author ORCID or name + affiliation
   → Filter to papers where author position = first
   → This is the MOST IMPORTANT component and is freely available

2. Funded research months:
   → NSF GSS: mechanism of support (RA/fellowship/TA) by year
   → NIH RePORTER: exact funding periods for T32/F31/F32
   → ProQuest dissertations: time-to-degree (proxy for research intensity)
   
3. Stipend:
   → NSF GSS doesn't report 

---
## Part 3: OpenAlex Queries for Publication-Based Metrics

OpenAlex is free, requires no API key (just a polite email in the header), and 
covers >250M works. We can use it to compute first-author publication rates for 
graduate students at different institutions.

In [6]:
# ============================================================
# OpenAlex API functions
# ============================================================

OPENALEX_BASE = "https://api.openalex.org"

def query_openalex(endpoint, params, email="your_email@temple.edu"):
    """Query OpenAlex API with polite pool access."""
    params['mailto'] = email  # gets you into the polite pool (faster)
    resp = requests.get(f"{OPENALEX_BASE}/{endpoint}", params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

def get_institution_id(name):
    """Look up OpenAlex institution ID by name."""
    data = query_openalex("institutions", {"search": name})
    if data.get('results'):
        inst = data['results'][0]
        return inst['id'], inst['display_name']
    return None, None

def get_works_by_institution(inst_id, from_year=2008, to_year=2024, 
                              author_position='first', per_page=200):
    """Get works where authors at an institution are in a specific position."""
    # OpenAlex filter syntax
    filters = [
        f"authorships.institutions.id:{inst_id}",
        f"publication_year:{from_year}-{to_year}",
        f"authorships.author_position:{author_position}",
        "type:article",  # journal articles only
    ]
    
    params = {
        "filter": ",".join(filters),
        "per_page": 1,  # just get the count
    }
    
    data = query_openalex("works", params)
    return data.get('meta', {}).get('count', 0)

In [7]:
# ============================================================
# Compare first-author publication rates across institutions
# ============================================================
# NOTE: This queries OpenAlex, which IS publicly accessible.
# If running locally, replace the email with your own.

INSTITUTIONS_TO_COMPARE = [
    "Temple University",
    "University of Pittsburgh",
    "Penn State University",
    "University of Pennsylvania",
    "Drexel University",
    "Rutgers University",
    "University of Michigan",
    "Yale University",
]

print("Querying OpenAlex for first-author publication counts...")
print("(Using polite pool - replace email with yours for faster access)")
print()

openalex_results = []

for inst_name in INSTITUTIONS_TO_COMPARE:
    try:
        inst_id, display_name = get_institution_id(inst_name)
        if inst_id:
            # Total articles
            total = get_works_by_institution(inst_id, 2015, 2024, 'first')
            # Last-author articles (proxy for PI-led group publications)
            last = get_works_by_institution(inst_id, 2015, 2024, 'last')
            
            openalex_results.append({
                'institution': display_name,
                'openalex_id': inst_id,
                'first_author_articles_2015_2024': total,
                'last_author_articles_2015_2024': last,
                'first_to_last_ratio': total / max(last, 1),
            })
            print(f"  {display_name}: {total:,} first-author, {last:,} last-author articles")
        time.sleep(0.5)
    except Exception as e:
        print(f"  {inst_name}: Error - {e}")

if openalex_results:
    openalex_df = pd.DataFrame(openalex_results)
    openalex_df.to_csv('data/openalex_institution_comparison.csv', index=False)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 5))
    openalex_df_sorted = openalex_df.sort_values('first_author_articles_2015_2024')
    ax.barh(openalex_df_sorted['institution'], 
            openalex_df_sorted['first_author_articles_2015_2024'],
            color=COLORS[2], alpha=0.8, label='First-author')
    ax.set_xlabel('Articles (2015-2024)')
    ax.set_title('First-Author Article Counts by Institution (all departments)')
    plt.tight_layout()
    plt.savefig('figures/openalex_first_author_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

Querying OpenAlex for first-author publication counts...
(Using polite pool - replace email with yours for faster access)

  Temple University: Error - 400 Client Error: Bad Request for url: https://api.openalex.org/works?filter=authorships.institutions.id%3Ahttps%3A%2F%2Fopenalex.org%2FI84392919%2Cpublication_year%3A2015-2024%2Cauthorships.author_position%3Afirst%2Ctype%3Aarticle&per_page=1&mailto=your_email%40temple.edu
  University of Pittsburgh: Error - 400 Client Error: Bad Request for url: https://api.openalex.org/works?filter=authorships.institutions.id%3Ahttps%3A%2F%2Fopenalex.org%2FI170201317%2Cpublication_year%3A2015-2024%2Cauthorships.author_position%3Afirst%2Ctype%3Aarticle&per_page=1&mailto=your_email%40temple.edu
  Penn State University: Error - 400 Client Error: Bad Request for url: https://api.openalex.org/works?filter=authorships.institutions.id%3Ahttps%3A%2F%2Fopenalex.org%2FI130769515%2Cpublication_year%3A2015-2024%2Cauthorships.author_position%3Afirst%2Ctype%3Aartic

---
## Part 4: Putting It Together — The Analysis Pipeline

Here's the complete pipeline for answering David's three questions. Each step 
is annotated with what data is needed and where to get it.

In [8]:
print("""
═══════════════════════════════════════════════════════════════
COMPLETE ANALYSIS PIPELINE
═══════════════════════════════════════════════════════════════

STEP 1: BUILD THE MASTER INSTITUTION-DEPARTMENT DATASET
────────────────────────────────────────────────────────
  a) Download NSF GSS public use files (2008-2024)
     → https://ncses.nsf.gov/explore-data/microdata/graduate-students-postdoctorates-s-e
     → Key variables: UNITID, institution, dept, grad students by mechanism
       (RA, fellowship, traineeship, TA, self-funded)
     → This gives you the DECOMPOSITION of "funded labor" that Zhang lacks
     
  b) Run Part 1 of this notebook (NIH RePORTER queries)
     → Gives T32/F31/F32 counts per institution-department
     
  c) Download IPEDS data for faculty counts
     → https://nces.ed.gov/ipeds/ → "Use the Data" → Complete Data Files
     → Key file: "Instructional Staff/Salaries" (faculty counts by rank)
     
  d) Use Zhang et al.'s prestige scores from Wapman et al. (2022)
     → https://github.com/LarremoreLab/us-faculty-hiring-networks
     → Maps institution → within-discipline prestige percentile

  MERGE KEY: UNITID (IPEDS) links to NSF GSS institution ID
             Institution name links to NIH RePORTER org_name
             Manual crosswalk needed for Zhang et al. (anonymized)

STEP 2: ANSWER QUESTION 1 (Temple's labor-productivity gap)
───────────────────────────────────────────────────────────
  Model: Productivity_dept ~ β₁·FundedLabor + β₂·Prestige + controls + ε
  
  Compute residual ε for each department.
  Plot residuals vs. training grant density.
  
  PREDICTION: Departments with HIGH funded labor but LOW T32/F31 density
  (i.e., students funded by intramural mechanisms with no research requirement)
  will have NEGATIVE residuals — they're less productive than expected.
  
  Temple's psychology dept is predicted to be in this quadrant.

STEP 3: ANSWER QUESTION 2 (decomposing funding sources)
──────────────────────────────────────────────────────
  Augmented model:
    Productivity ~ β₁·RA_funded + β₂·T32_funded + β₃·F_funded 
                 + β₄·Fellowship_funded + β₅·Prestige + controls
  
  The NSF GSS mechanism-of-support breakdown gives:
    - Research Assistantship → maps to RA_funded
    - Fellowship → maps to Fellowship_funded (intramural) + F_funded (from RePORTER)
    - Traineeship → maps to T32_funded (from RePORTER)
    - Teaching Assistantship → "unfunded" in Zhang et al.
    - Self-funded → "unfunded" in Zhang et al.
    
  To separate intramural from extramural fellowships:
    Fellowship_intramural = GSS_fellowship_count - RePORTER_F31_F32_count
    (This is an approximation; some fellowships are from NSF, DOD, etc.)

STEP 4: ANSWER QUESTION 3 (quality-weighted doctorates)
──────────────────────────────────────────────────────
  For each institution's doctoral graduates:
    a) Count from NCSES Earned Doctorates survey (already in NCSES profiles)
    b) Link to OpenAlex for first-author publication counts
    c) Link to NIH RePORTER for training grant funding status
    d) Compute RWD = Σ (first_author_pubs × funding_weight)
    
  Compare: Carnegie R1 status vs. RWD ranking
  Identify institutions that are R1 by headcount but would NOT be by RWD.
  
═══════════════════════════════════════════════════════════════
ESTIMATED TIMELINE:
  Week 1: Download NSF GSS + IPEDS + run this notebook locally
  Week 2: Build crosswalks, merge datasets
  Week 3: Run the decomposition regressions (Steps 2-3)
  Week 4: OpenAlex queries for RWD metric (Step 4)
  Week 5: Write up, visualize, interpret
═══════════════════════════════════════════════════════════════
""")


═══════════════════════════════════════════════════════════════
COMPLETE ANALYSIS PIPELINE
═══════════════════════════════════════════════════════════════

STEP 1: BUILD THE MASTER INSTITUTION-DEPARTMENT DATASET
────────────────────────────────────────────────────────
  a) Download NSF GSS public use files (2008-2024)
     → https://ncses.nsf.gov/explore-data/microdata/graduate-students-postdoctorates-s-e
     → Key variables: UNITID, institution, dept, grad students by mechanism
       (RA, fellowship, traineeship, TA, self-funded)
     → This gives you the DECOMPOSITION of "funded labor" that Zhang lacks
     
  b) Run Part 1 of this notebook (NIH RePORTER queries)
     → Gives T32/F31/F32 counts per institution-department
     
  c) Download IPEDS data for faculty counts
     → https://nces.ed.gov/ipeds/ → "Use the Data" → Complete Data Files
     → Key file: "Instructional Staff/Salaries" (faculty counts by rank)
     
  d) Use Zhang et al.'s prestige scores from Wapman et al. (20

---
## Part 5: What We Can Already Say About Temple

Even without running the full pipeline, the NCSES data tells us a lot.

In [9]:
print("""
TEMPLE UNIVERSITY — POSITION IN THE LABOR ADVANTAGE FRAMEWORK
══════════════════════════════════════════════════════════════

FROM NCSES INSTITUTIONAL PROFILE (2024 data):
  • R&D expenditure ranking: 107th (88.6th percentile among 925 institutions)
  • Full-time grad students ranking: 137th (78.6th percentile among 631)
  • Earned doctorates ranking: 81st (82.7th percentile among 459)
  • Research space ranking: ~142nd (76.8th percentile)

PSYCHOLOGY SPECIFICALLY (from NCSES GSS):
  • ~214 full-time psychology grad students (2024)
  • ~32 doctoral faculty (from Temple website)
  • Rough student-to-faculty ratio: ~6.7
  • Temple awards ~19-24 psychology PhDs per year
  
PLACING TEMPLE IN ZHANG ET AL.'S FRAMEWORK:
  • Prestige: roughly 6th-7th decile (upper middle)
  • For psychology at this prestige level, Zhang et al. Fig. 1C shows
    ~1-2 funded researchers per faculty member
  • Temple's student/faculty ratio (6.7) is ABOVE the median for psychology
    (median ~5 per faculty from Table S13), BUT...
  • ...the critical question is: how many of those 6.7 students per faculty
    are doing RESEARCH vs. TEACHING vs. COURSEWORK?
    
THE DAVID SMITH HYPOTHESIS:
  If Temple's funded students have no research expectations, then:
  • NSF GSS counts them as "funded" (inflating the funded labor ratio)
  • Zhang et al.'s model predicts Temple should be MORE productive
  • But actual productivity is LOWER than predicted
  • The residual = (actual - predicted) is NEGATIVE
  • This makes Temple a case study for the "inefficiency" in Extension 4

TO TEST THIS: Run the full pipeline above. The prediction is specific
and falsifiable: Temple psychology will show a negative residual in the
funded labor → productivity regression, and this residual will be
partially explained by the absence of T32 funding and low F31/F32 rates
compared to peer institutions.
""")


TEMPLE UNIVERSITY — POSITION IN THE LABOR ADVANTAGE FRAMEWORK
══════════════════════════════════════════════════════════════

FROM NCSES INSTITUTIONAL PROFILE (2024 data):
  • R&D expenditure ranking: 107th (88.6th percentile among 925 institutions)
  • Full-time grad students ranking: 137th (78.6th percentile among 631)
  • Earned doctorates ranking: 81st (82.7th percentile among 459)
  • Research space ranking: ~142nd (76.8th percentile)

PSYCHOLOGY SPECIFICALLY (from NCSES GSS):
  • ~214 full-time psychology grad students (2024)
  • ~32 doctoral faculty (from Temple website)
  • Rough student-to-faculty ratio: ~6.7
  • Temple awards ~19-24 psychology PhDs per year
  
PLACING TEMPLE IN ZHANG ET AL.'S FRAMEWORK:
  • Prestige: roughly 6th-7th decile (upper middle)
  • For psychology at this prestige level, Zhang et al. Fig. 1C shows
    ~1-2 funded researchers per faculty member
  • Temple's student/faculty ratio (6.7) is ABOVE the median for psychology
    (median ~5 per faculty from

In [10]:
# Quick computation with what we have from Zhang et al.'s data
if area_strict is not None:
    psych = area_strict[area_strict['Area'] == 'Psychological Sciences'].copy()
    print(f"Psychology departments in Zhang et al. strict data: {len(psych)}")
    print(f"\nDescriptive statistics:")
    print(f"  Productivity: mean={psych['Productivity'].mean():.2f}, "
          f"median={psych['Productivity'].median():.2f}")
    print(f"  Funded labor ratio: mean={psych['funded_per_faculty'].mean():.2f}, "
          f"median={psych['funded_per_faculty'].median():.2f}")
    print(f"  Group size: mean={psych['WindowedGroupSize'].mean():.2f}, "
          f"median={psych['WindowedGroupSize'].median():.2f}")
    print(f"  Prestige range: {psych['uniform_percentile100'].min():.0f} to "
          f"{psych['uniform_percentile100'].max():.0f}")
    
    # Where would Temple fall? (~78th percentile prestige)
    temple_approx_prestige = 78
    nearby = psych[(psych['uniform_percentile100'] > 70) & 
                   (psych['uniform_percentile100'] < 85)]
    print(f"\n  Psychology depts at Temple's approximate prestige level (70-85th %ile):")
    print(f"    N = {len(nearby)}")
    print(f"    Mean productivity: {nearby['Productivity'].mean():.2f}")
    print(f"    Mean funded labor: {nearby['funded_per_faculty'].mean():.2f}")
    print(f"    Mean group size: {nearby['WindowedGroupSize'].mean():.2f}")
    
    # Compare to top decile
    top = psych[psych['uniform_percentile100'] > 90]
    print(f"\n  Top-decile psychology departments (>90th %ile):")
    print(f"    N = {len(top)}")
    print(f"    Mean productivity: {top['Productivity'].mean():.2f}")
    print(f"    Mean funded labor: {top['funded_per_faculty'].mean():.2f}")
    print(f"    Mean group size: {top['WindowedGroupSize'].mean():.2f}")

Psychology departments in Zhang et al. strict data: 55

Descriptive statistics:
  Productivity: mean=1.90, median=1.90
  Funded labor ratio: mean=1.44, median=0.96
  Group size: mean=4.27, median=4.43
  Prestige range: 4 to 98

  Psychology depts at Temple's approximate prestige level (70-85th %ile):
    N = 7
    Mean productivity: 2.48
    Mean funded labor: 2.04
    Mean group size: 4.76

  Top-decile psychology departments (>90th %ile):
    N = 3
    Mean productivity: 3.07
    Mean funded labor: 2.10
    Mean group size: 6.07


In [15]:
print("""
NEXT STEPS FOR DAVID:
═════════════════════
1. Run this notebook locally (it will hit the NIH RePORTER and OpenAlex APIs)
2. Download NSF GSS public use files from:
   https://ncses.nsf.gov/explore-data/microdata/graduate-students-postdoctorates-s-e
3. The GSS files have the mechanism-of-support breakdown that lets us
   separate RA-funded from fellowship-funded from TA-funded students
4. Once we have that decomposition, I can build the augmented regression
   model that tests whether the labor advantage disappears when you
   account for WHAT TYPE of funding students receive

The argument you're building is powerful:
  "It's not just about having more money — it's about whether funded 
   students are expected (and supported) to do research."
   
This reframes the labor advantage from a simple resource story to a 
STRUCTURAL/POLICY story about how institutions deploy their graduate 
student funding.
""")


NEXT STEPS FOR DAVID:
═════════════════════
1. Run this notebook locally (it will hit the NIH RePORTER and OpenAlex APIs)
2. Download NSF GSS public use files from:
   https://ncses.nsf.gov/explore-data/microdata/graduate-students-postdoctorates-s-e
3. The GSS files have the mechanism-of-support breakdown that lets us
   separate RA-funded from fellowship-funded from TA-funded students
4. Once we have that decomposition, I can build the augmented regression
   model that tests whether the labor advantage disappears when you
   account for WHAT TYPE of funding students receive

The argument you're building is powerful:
  "It's not just about having more money — it's about whether funded 
   students are expected (and supported) to do research."

This reframes the labor advantage from a simple resource story to a 
STRUCTURAL/POLICY story about how institutions deploy their graduate 
student funding.

